In [2]:
import pandas as pd
import numpy as np
import re

In [4]:
with open("parsed_result.md", "r", encoding="utf-8") as f:
    lines = f.readlines()

parsed_data = []

In [5]:
# 초기값 세팅
current_year = 2021.0
current_phase = "1차"
current_complex = "알수없음"
current_stage = "최종" # 기본값을 '최종'으로 설정

In [7]:
for line in lines:
    line = line.strip()
    # 연도 및 차수, 심사단계(서류/최종) 추적
    if line.startswith("# "):
        year_match = re.search(r'(20\d{2})(?:년|\-)?\s*(\d)차?', line)
        if year_match:
            current_year = float(year_match.group(1))
            current_phase = f"{year_match.group(2)}차"

            if '서류심사' in line or '커트라인' in line:
                currnet_stage = "서류"
            elif '당첨자' in line or '합격선' in line:
                current_stage = "최종"
            continue
    if not line.startswith('|'):
        continue

    cells = [c.strip() for c in line.split('|')[1:-1]]
    if not cells: continue

    line_text = " ".join(cells)

    # 단지명 상속
    for cell in cells[:2]:
        if len(cell) >= 2 and not re.match(r'^\d', cell) and not any(x in cell for x in ['행복주택', '일반', '우선', '신혼부부', '청년', '고령자', '대학생', '주거급여']):
            current_complex = cell
            break
                
    # 점수 스캔
    score_match = re.search(r'(\d+(?:\.\d+)?)\s*점', line_text)
    if score_match:
        score = float(score_match.group(1))

        # 공급유형(평수) 추출
        size = "39" # 기본값을 39로 설정
        size_match = re.search(r'(?<!\d)([1-9]\d)[a-zA-Z가-힣]?(?!\d|\s*점|\s*순위)', line_text)
        if size_match:
            size = size_match.group(1)

        # 신청자격 정제
        elig = '일반가구'
        for keyword in ['청년', '신혼부부', '고령자', '대학생', '주거급여']:
            if keyword in line_text:
                elig = keyword
                break
        
        # 당첨순위 스캔
        rank_match = re.search(r'(\d+)순위', line_text)
        rank = int(rank_match.group(1)) if rank_match else 1

        parsed_data.append({
            '연도': current_year,
            '차수': current_phase,
            '단지명': current_complex,
            '공급유형': size,
            '신청자격': elig,
            '당첨순위': rank,
            '심사단계': current_stage,
            '커트라인점수': score
        })


In [8]:
# 데이터프레임 생성 및 정제
final_df = pd.DataFrame(parsed_data)

In [9]:
final_df['단지명'] = final_df['단지명'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

print(f"추출 완료. 총 {len(final_df)}줄의 데이터를 성공적으로 규격화했습니다.")
display(final_df.head(10))

추출 완료. 총 1146줄의 데이터를 성공적으로 규격화했습니다.


,연도,차수,단지명,공급유형,신청자격,당첨순위,심사단계,커트라인점수
0,2021.0,1차,디에이치 포레센트,20,고령자,1,최종,6.0
1,2021.0,1차,디에이치 포레센트,39,일반가구,1,최종,6.0
2,2021.0,1차,르엘대치,39,고령자,1,최종,5.0
3,2021.0,1차,르엘대치,10,신혼부부,1,최종,6.0
4,2021.0,1차,보라매자이,14,고령자,1,최종,6.0
5,2021.0,1차,보라매자이,11,신혼부부,1,최종,6.0
6,2021.0,1차,보라매자이,39,고령자,1,최종,6.0
7,2021.0,1차,보라매자이,26,신혼부부,1,최종,6.0
8,2021.0,1차,------------,27,고령자,1,최종,5.0
9,2021.0,1차,------------,39,신혼부부,2,최종,4.0


In [10]:
import pandas as pd
import re
import unittest
from typing import List, Dict, Any

In [11]:
def extract_housing_data(file_path: str) -> pd.DataFrame:
    """
    마크다운 파일을 파싱하여 청년안심주택/행복주택 규격에 맞는 데이터프레임으로 반환합니다.
    """
    parsed_data: List[Dict[str, Any]] = []
    
    current_year = 2021.0
    current_phase = "1차"
    current_complex = "알수없음"
    current_stage = "최종"
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: '{file_path}' 파일을 찾을 수 없습니다.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error: 파일 읽기 중 오류 발생 - {e}")
        return pd.DataFrame()

    for line in lines:
        line = line.strip()
        
        # 연도, 차수, 심사단계 추적
        if line.startswith('#'):
            year_match = re.search(r'(20\d{2})(?:년|\-)?\s*(\d)차?', line)
            if year_match:
                current_year = float(year_match.group(1))
                current_phase = f"{year_match.group(2)}차"
            
            if '서류심사' in line or '커트라인' in line:
                current_stage = "서류"
            elif '당첨자' in line or '합격선' in line:
                current_stage = "최종"
            continue
            
        # 표 데이터가 아니거나 마크다운 구분선(|---|)인 경우 스킵
        if not line.startswith('|') or '---' in line:
            continue
            
        cells = [c.strip() for c in line.split('|')[1:-1]]
        if not cells:
            continue
        
        line_text = " ".join(cells)

        # 단지명 갱신 (마크다운 찌꺼기 방어)
        for cell in cells[:2]:
            if len(cell) >= 2 and not re.match(r'^[\d\-]+$', cell) and not any(x in cell for x in ['행복주택', '일반', '우선', '신혼부부', '청년', '고령자', '대학생', '주거급여', '합계', '소계']):
                current_complex = cell
                break

        # 점수 추출
        score_match = re.search(r'(\d+(?:\.\d+)?)\s*점', line_text)
        if score_match:
            score = float(score_match.group(1))
            
            # 공급유형 추출 (두 자리 이상의 숫자, 점수나 순위 제외)
            size = "39"
            size_match = re.search(r'(?<!\d)([1-9]\d{1,2})[a-zA-Z가-힣]?(?!\d|\s*점|\s*순위)', line_text)
            if size_match:
                size = size_match.group(1)
                
            # 신청자격 추출
            elig = '일반가구'
            for keyword in ['청년', '신혼부부', '고령자', '대학생', '주거급여']:
                if keyword in line_text:
                    elig = keyword
                    break
                    
            # 당첨순위 추출
            rank_match = re.search(r'(\d+)순위', line_text)
            rank = int(rank_match.group(1)) if rank_match else 1

            parsed_data.append({
                '연도': current_year,
                '차수': current_phase,
                '단지명': re.sub(r'\(.*?\)', '', current_complex).strip(),
                '공급유형': size,
                '신청자격': elig,
                '당첨순위': rank,
                '심사단계': current_stage,
                '커트라인점수': score
            })

    return pd.DataFrame(parsed_data)

In [12]:
# 실행 부
if __name__ == "__main__":
    final_df = extract_housing_data("parsed_result.md")
    
    if not final_df.empty:
        # 중복 데이터 제거
        final_df = final_df.drop_duplicates().reset_index(drop=True)
        display(final_df.head(10))

,연도,차수,단지명,공급유형,신청자격,당첨순위,심사단계,커트라인점수
0,2021.0,1차,디에이치 포레센트,20,고령자,1,최종,6.0
1,2021.0,1차,디에이치 포레센트,39,일반가구,1,최종,6.0
2,2021.0,1차,르엘대치,39,고령자,1,최종,5.0
3,2021.0,1차,르엘대치,10,신혼부부,1,최종,6.0
4,2021.0,1차,보라매자이,14,고령자,1,최종,6.0
5,2021.0,1차,보라매자이,11,신혼부부,1,최종,6.0
6,2021.0,1차,보라매자이,39,고령자,1,최종,6.0
7,2021.0,1차,보라매자이,26,신혼부부,1,최종,6.0
8,2021.0,1차,주택 유형,27,고령자,1,최종,5.0
9,2021.0,1차,주택 유형,39,신혼부부,2,최종,4.0


In [13]:
class TestHousingParser(unittest.TestCase):
    def test_regex_matching(self):
        sample_line = "우선 1순위 6점 (자치구 전입일 : 1987.03.20)"
        
        # 점수 추출 테스트
        score_match = re.search(r'(\d+(?:\.\d+)?)\s*점', sample_line)
        self.assertIsNotNone(score_match)
        self.assertEqual(float(score_match.group(1)), 6.0)
        
        # 순위 추출 테스트
        rank_match = re.search(r'(\d+)순위', sample_line)
        self.assertIsNotNone(rank_match)
        self.assertEqual(int(rank_match.group(1)), 1)